# EpiRank modern example

This notebook uses the modernized modules and is compatible with Python 3.10 and NetworkX 3.x.

## Imports

In [4]:
from pathlib import Path
import sys

import pandas as pd


def find_repo_root(start):
    for path in [start, *start.parents]:
        if (path / "data_flow").exists() and (path / "scripts" / "EpiRank").exists():
            return path
    raise RuntimeError("Could not locate Net_Dyns_Project_EpiRank repo root")


repo_root = find_repo_root(Path.cwd().resolve())
scripts_dir = repo_root / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

from EpiRank import epirank_modern as epirank
from EpiRank import additional_analysis_modern as aa

## Load origin-destination flow data

In [6]:
print(f"Repo root: {repo_root}")
df = pd.read_csv(repo_root / "data_flow" / "od_flow_data.csv", index_col=0)
df.head()

Repo root: /Users/gqs/Documents/Net_Dyns/Project/Net_Dyns_Project_EpiRank


,origin,destination,flow
ind,,,
0,p02,s08,2.0
1,m14,f13,3.0
2,g10,r27,5.0
3,n16,n16,24824.0
4,i15,i10,1455.0


## Build a weighted directed graph

In [7]:
g = epirank.make_DiGraph(
    df,
    origin_col="origin",
    destination_col="destination",
    flow_col="flow",
    largest_connected_component=False,
    exclude_selfloop=False,
)

g.number_of_nodes(), g.number_of_edges()

graph construction done, no. nodes: 353, no. edges: 11220


(353, 11220)

## Calculate EpiRank

In [8]:
epi_vals05 = epirank.run_epirank(g, daytime=0.5, d=0.95)

(
    pd.DataFrame.from_dict(epi_vals05, orient="index", columns=["epirank"])
    .sort_values("epirank", ascending=False)
    .head(10)
)

start preparing matrices
preparation done, start iterating
epirank calculation done after iteration: 581


,epirank
g01,0.011191
r17,0.010291
j02,0.010250
m09,0.009618
f22,0.009598
p19,0.008704
d00,0.008168
l03,0.007843
u05,0.007733
c10,0.007348


## Baseline metrics

In [9]:
baseline_metrics = aa.calculate_metrices(g, d=0.95)
{name: list(values.items())[:3] for name, values in baseline_metrics}

{'pagerank': [('p02', 0.0016103223171920937),
  ('s08', 0.0007350189740969518),
  ('m14', 0.0005916744882971193)],
 'hub_rank': [('p02', 3.117597075621531e-18),
  ('s08', 9.334521229628576e-19),
  ('m14', 1.2606080866610459e-20)],
 'authority_rank': [('p02', -0.0),
  ('s08', 9.833632877184666e-20),
  ('m14', -2.8603568652489714e-22)]}